# E5 · Estudio de caso Luna–Sunset–Cat-8

**Objetivo.** Examinar si el re-ranking morfológico E5 corrige confusiones visuales de E4 entre tres gatos similares de PetTrace v6.  
**Pregunta.** ¿E5 recupera consultas de Luna confundidas por E4 con Sunset o Cat-8 sin dañar aciertos de Luna?  
**Hipótesis.** La doble compuerta solo cambiará casos visualmente ambiguos cuando exista una ventaja morfológica suficiente.  
**Intervención y control.** E5 congelado frente al E4 congelado, ambos provenientes de la misma corrida publicada.  
**Muestra.** Las 30 consultas de Luna, Sunset y Cat-8 dentro del test inmutable PetTrace v6.  
**Protocolo.** Análisis explicativo posterior a la evaluación oficial; no ajusta umbrales, no reentrena y no modifica las métricas agregadas de E4/E5.  
**Métricas.** Aciertos Rank-1, consultas rescatadas y consultas dañadas.


## Fase 1. Evidencia congelada

Se descargan únicamente el resultado oficial de E5 y cuatro imágenes de PetTrace v6. No se ejecutan CLIP, LoRA, la MLP ni Qwen nuevamente.

In [ ]:
%pip install -q --no-cache-dir "pettrace-ml[tracking,notebooks]==0.2.46"

In [ ]:
import csv
import hashlib
import json
import os
from pathlib import Path

from google.colab import userdata
from huggingface_hub import hf_hub_download
from pettrace.tracking import tracking_context

for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Falta habilitar el secreto {name} para este notebook")
    os.environ[name] = value
os.environ["MLFLOW_TRACKING_URI"] = "https://ml.pettrace.app"

E5_RUN = "57ffea8d48a64561be3651fad9b0d914"
DATASET_REPO = "alessandro54/pettrace-reid"
DATASET_VERSION = "v6"
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
REPORT = Path("/content/reports/e5-luna-hard-case")
REPORT.mkdir(parents=True, exist_ok=True)

client, experiment = tracking_context()
source_run = client.get_run(E5_RUN)
if source_run.info.status != "FINISHED":
    raise RuntimeError("La corrida E5 oficial no está finalizada")

def load_run_json(path):
    local = client.download_artifacts(E5_RUN, path, str(REPORT / "source"))
    return json.loads(Path(local).read_text())

evaluation = load_run_json("report/evaluation-input.json")
calibration = load_run_json("report/calibration.json")
dataset = evaluation["dataset"]
if (dataset["repo"], dataset["version"], dataset["commit"]) != (
    DATASET_REPO, DATASET_VERSION, DATASET_COMMIT
):
    raise RuntimeError("La evidencia no corresponde al snapshot PetTrace v6 congelado")
if calibration["selection_split"] != "validation" or calibration["test_sets_read"]:
    raise RuntimeError("La calibración no conserva la separación validación/test")

metadata_path = hf_hub_download(
    DATASET_REPO, "metadata.csv", repo_type="dataset",
    revision=DATASET_COMMIT, token=os.environ["HF_TOKEN"]
)
with open(metadata_path, newline="") as handle:
    metadata = list(csv.DictReader(handle))
if len(metadata) != 502 or evaluation["result"]["test"]["global"]["queries"] != 502:
    raise RuntimeError("La población de test difiere del control oficial")
print("Evidencia verificada · E5 oficial · PetTrace v6 · 502 consultas")

## Fase 2. Análisis del caso difícil

Se comparan los Top-1 ya publicados. Las imágenes de Sunset y Cat-8 son referencias visuales de identidad; no se presentan como la fotografía exacta que maximizó cada similitud.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from PIL import Image
from pettrace.ui import markdown_table

IDENTITIES = {
    "Luna": "cat-b8d7189e",
    "Sunset": "cat-54af8927",
    "Cat-8": "cat-09843e72",
}
rows = evaluation["result"]["test"]["per_query"]
case_rows = [row for row in rows if row["animal"] in IDENTITIES.values()]
if len(case_rows) != 30:
    raise RuntimeError("El subconjunto esperado debe contener 30 consultas")

summary_rows = []
case_summary = {}
for label, animal_id in IDENTITIES.items():
    identity_rows = [row for row in case_rows if row["animal"] == animal_id]
    values = {
        "queries": len(identity_rows),
        "e4_correct": sum(row["baseline_top1"] == animal_id for row in identity_rows),
        "e5_correct": sum(row["reranked_top1"] == animal_id for row in identity_rows),
        "rescued": sum(row["rescued"] for row in identity_rows),
        "damaged": sum(row["damaged"] for row in identity_rows),
    }
    case_summary[label] = values
    summary_rows.append([label, *values.values()])

luna = case_summary["Luna"]
if luna != {"queries": 10, "e4_correct": 2, "e5_correct": 4, "rescued": 2, "damaged": 0}:
    raise RuntimeError("El caso Luna no coincide con la evidencia oficial congelada")
rescued = [row for row in case_rows if row["animal"] == IDENTITIES["Luna"] and row["rescued"]]
if [row["query"] for row in rescued] != [55, 56]:
    raise RuntimeError("Las consultas rescatadas de Luna cambiaron")

table = markdown_table(
    ["Identidad", "Consultas", "E4 correcto", "E5 correcto", "Rescatadas", "Dañadas"],
    summary_rows,
)
display(Markdown(table))

selected_queries = [55, 56, 148, 16]
titles = [
    "Luna · consulta 55 · E4→Sunset, E5→Luna",
    "Luna · consulta 56 · E4→Sunset, E5→Luna",
    "Sunset · imagen representativa",
    "Cat-8 · imagen representativa",
]
images = []
for query in selected_queries:
    item = metadata[query]
    local = hf_hub_download(
        DATASET_REPO, item["file_name"], repo_type="dataset",
        revision=DATASET_COMMIT, token=os.environ["HF_TOKEN"]
    )
    images.append(Image.open(local).convert("RGB"))

fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for axis, image, title in zip(axes.flat, images, titles):
    axis.imshow(image)
    axis.set_title(title, fontsize=10)
    axis.axis("off")
fig.suptitle("E5 · estudio de caso Luna–Sunset–Cat-8", fontsize=14)
fig.tight_layout()
figure_path = REPORT / "luna-hard-case.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()

focused_e4 = sum(row["baseline_top1"] == row["animal"] for row in case_rows)
focused_e5 = sum(row["reranked_top1"] == row["animal"] for row in case_rows)
result = {
    "study_type": "post_hoc_explanatory_case_study",
    "official_e5_run": E5_RUN,
    "dataset": dataset,
    "frozen_thresholds": calibration["selected"],
    "identities": case_summary,
    "focused_subset": {"queries": 30, "e4_correct": focused_e4, "e5_correct": focused_e5},
    "luna_rescued_queries": [
        {key: row[key] for key in ("query", "baseline_top1", "reranked_top1", "visual_margin", "text_advantage")}
        for row in rescued
    ],
    "limitations": [
        "The subset was inspected after the official benchmark and is not a selection set.",
        "E5 does not fully solve Luna: 6 of 10 Luna queries remain incorrect at Rank-1.",
        "The missing-eye annotation is human-curated metadata; this study does not claim that Qwen detected it.",
    ],
}
(REPORT / "case-results.json").write_text(json.dumps(result, indent=2) + "\n")
with open(REPORT / "case-table.csv", "w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["identity", "queries", "e4_correct", "e5_correct", "rescued", "damaged"])
    writer.writerows(summary_rows)
display(Markdown(
    "**Lectura.** E5 recuperó 2 de las 10 consultas de Luna que E4 confundía con Sunset, "
    "sin dañar un acierto de Luna. El caso no está completamente resuelto: 6 consultas de Luna "
    "siguen fallando en Rank-1."
))

## Fase 3. Publicación del estudio de caso

La evidencia se registra como una corrida explicativa separada. No registra otra versión del modelo ni altera las corridas oficiales.

In [ ]:
from pettrace.tracking import tracking_run_url

identity = {
    "parent_run": E5_RUN,
    "dataset_commit": DATASET_COMMIT,
    "scope": sorted(IDENTITIES.values()),
    "thresholds": calibration["selected"],
}
publication_key = hashlib.sha256(
    json.dumps(identity, sort_keys=True).encode()
).hexdigest()
tag = "pettrace.case_study_key"
matches = client.search_runs(
    [experiment.experiment_id],
    filter_string=f"tags.`{tag}` = '{publication_key}'",
    max_results=2,
)
if len(matches) > 1:
    raise RuntimeError("Existen múltiples corridas para este estudio de caso")
run = matches[0] if matches else client.create_run(
    experiment.experiment_id,
    tags={
        tag: publication_key,
        "mlflow.runName": "E5 · Luna–Sunset hard-case study on pettrace-reid@v6",
        "purpose": "post_hoc_explanatory_case_study",
        "selection": "no_tuning",
        "parent_e5_run": E5_RUN,
        "dataset": "pettrace-reid@v6",
        "dataset_commit": DATASET_COMMIT,
    },
)
run_id = run.info.run_id
if run.info.status != "FINISHED":
    for key, value in {
        "tau_v": calibration["selected"]["tau_v"],
        "lambda": calibration["selected"]["lambda"],
        "tau_t": calibration["selected"]["tau_t"],
        "queries": 30,
    }.items():
        client.log_param(run_id, key, value)
    for key, value in {
        "luna.e4_rank1": luna["e4_correct"] / luna["queries"],
        "luna.e5_rank1": luna["e5_correct"] / luna["queries"],
        "luna.rescued": luna["rescued"],
        "luna.damaged": luna["damaged"],
        "focused.e4_rank1": focused_e4 / 30,
        "focused.e5_rank1": focused_e5 / 30,
    }.items():
        client.log_metric(run_id, key, value)
    for artifact in ("case-results.json", "case-table.csv", "luna-hard-case.png"):
        client.log_artifact(run_id, str(REPORT / artifact), artifact_path="case-study")
    client.set_terminated(run_id, status="FINISHED")

receipt = {
    "published": True,
    "run_id": run_id,
    "url": tracking_run_url(experiment.experiment_id, run_id),
    "publication_key": publication_key,
}
(REPORT / "publication.json").write_text(json.dumps(receipt, indent=2) + "\n")
display(Markdown(f"**Estudio publicado:** [{run_id}]({receipt['url']})"))